# Day 2-03｜Roboflow BBOX Dataset 與 YOLO Detection 訓練
> Python 籃球運動資料分析課程  
> 本單元使用 Day 1 BBOX 作業的 Roboflow YOLO export 格式，提供可執行的 Ultralytics 訓練程式，並用已訓練模型對參考影片輸出 preview。  
> 修課背景：具備基礎 Python 語法即可；不預設電腦視覺或運動資料分析經驗。

## 學習目標
- 確認 Roboflow detection dataset 的 `data.yaml` 與 images/labels 結構。
- 閱讀可執行的 YOLO detection 訓練程式。
- 使用已訓練 detector 對參考影片產生 preview。


## 執行階段提醒
請優先使用 **GPU** 或 **TPU** 的執行階段；不要使用純 CPU 執行。  
YOLO、MediaPipe 與影片處理在純 CPU 上會明顯較慢，容易讓課堂操作卡住。


## 資料放置位置
- Day 1 BBOX 作業 Roboflow YOLO export：`assets/datasets/roboflow_bbox_yolo/`
- 已訓練 detector 權重：`assets/models/detectors/yolo26n_basketball_player_best.pt`
- 訓練輸出：`assets/results/training/bbox_detection/`
- Preview 輸出：`assets/results/d2_03_trained_detector_preview.mp4`

## 學生下載提醒
1. 先在 Roboflow 網頁完成 bbox 標註。
2. 到 `Versions` 頁面按 `Generate New Version` / `Publish`。
3. 將新的 `workspace slug`、`project slug`、`version number` 填回下面的下載 cell。
4. `ROBOFLOW_API_KEY` 可留空，執行時會用 `getpass()` 輸入。


In [ ]:
from pathlib import Path
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/henry753951/basketball-hackathon-course.git"
REPO_BRANCH = "main"
IN_COLAB = False
DRIVE_MOUNTED = False

try:
    from google.colab import drive  # type: ignore[import-not-found]

    IN_COLAB = True
except ModuleNotFoundError:
    drive = None

if IN_COLAB and drive is not None:
    try:
        drive.mount("/content/drive")
        DRIVE_MOUNTED = True
    except NotImplementedError:
        print("目前這個 Colab runtime 不支援 google.colab.drive.mount，改用 /content 本機路徑。")

bootstrap_candidates = [
    Path.cwd().resolve(),
    *Path.cwd().resolve().parents,
    Path("/content/basketball_hackathon/course"),
    Path("/content/basketball_hackathon_course"),
    Path("/content/drive/MyDrive/basketball_hackathon/course"),
]

COURSE_ROOT_HINT = None
for candidate in bootstrap_candidates:
    if (candidate / "src" / "course_setup.py").exists():
        COURSE_ROOT_HINT = candidate
        break

if COURSE_ROOT_HINT is None and IN_COLAB:
    if DRIVE_MOUNTED:
        COURSE_ROOT_HINT = Path("/content/drive/MyDrive/basketball_hackathon/course")
    else:
        COURSE_ROOT_HINT = Path("/content/basketball_hackathon/course")

    if not (COURSE_ROOT_HINT / "src" / "course_setup.py").exists():
        if COURSE_ROOT_HINT.exists():
            shutil.rmtree(COURSE_ROOT_HINT)
        COURSE_ROOT_HINT.parent.mkdir(parents=True, exist_ok=True)
        cmd = [
            "git",
            "clone",
            "--depth",
            "1",
            "-b",
            REPO_BRANCH,
            REPO_URL,
            str(COURSE_ROOT_HINT),
        ]
        print("$", " ".join(cmd))
        subprocess.run(cmd, check=True)

if COURSE_ROOT_HINT is None or not (COURSE_ROOT_HINT / "src" / "course_setup.py").exists():
    raise FileNotFoundError(
        "找不到 src/course_setup.py。請先執行 init_colab.ipynb，或確認課程 repo 已經存在於目前目錄、/content/basketball_hackathon/course 或 Google Drive。"
    )

if str(COURSE_ROOT_HINT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT_HINT))

from src.course_setup import bootstrap_course_notebook  # noqa: E402

COURSE_ROOT = bootstrap_course_notebook(COURSE_ROOT_HINT, mount_drive=True)


In [ ]:
from src.roboflow_utils import dataset_status

BBOX_DATASET_DIR = COURSE_ROOT / "assets" / "datasets" / "roboflow_bbox_yolo"
MODEL_PATH = COURSE_ROOT / "assets" / "models" / "detectors" / "yolo26n_basketball_player_best.pt"

print(dataset_status(BBOX_DATASET_DIR))
print("model exists:", MODEL_PATH.exists(), MODEL_PATH)


In [ ]:
from getpass import getpass
from src.roboflow_utils import ensure_roboflow_detection_dataset

# 學生作業建議流程：先在 Roboflow 發布新 version，再把 version number 填在下面。
USE_ROBOFLOW_DOWNLOAD = False
ROBOFLOW_WORKSPACE = "YOUR_WORKSPACE"
ROBOFLOW_PROJECT = "YOUR_PROJECT"
ROBOFLOW_VERSION = 1
ROBOFLOW_API_KEY = ""
FORCE_DOWNLOAD = False

if USE_ROBOFLOW_DOWNLOAD:
    api_key = ROBOFLOW_API_KEY or getpass("Roboflow API key: ")
    data_yaml = ensure_roboflow_detection_dataset(
        BBOX_DATASET_DIR,
        workspace=ROBOFLOW_WORKSPACE,
        project=ROBOFLOW_PROJECT,
        version=ROBOFLOW_VERSION,
        api_key=api_key,
        export_format="yolov8",
        overwrite=FORCE_DOWNLOAD,
    )
    print("ready data.yaml:", data_yaml)
else:
    existing = BBOX_DATASET_DIR / "data.yaml"
    if existing.exists():
        print("YOLO detection dataset already exists:", existing)
    else:
        print("尚未找到 bbox dataset。可手動放入 YOLO export，或先在 Roboflow 發布新 version，再填入 API 設定並把 USE_ROBOFLOW_DOWNLOAD 改成 True。")


In [ ]:
RUN_TRAINING = False

if RUN_TRAINING:
    from ultralytics import YOLO

    data_yaml = BBOX_DATASET_DIR / "data.yaml"
    if not data_yaml.exists():
        raise FileNotFoundError(
            "找不到 data.yaml。請先將 Roboflow YOLO detection export 放到 assets/datasets/roboflow_bbox_yolo/"
        )

    model = YOLO("yolo26n.pt")
    results = model.train(
        data=str(data_yaml),
        epochs=100,
        imgsz=960,
        batch=2,
        workers=4,
        patience=30,
        close_mosaic=10,
        project=str(COURSE_ROOT / "assets" / "results" / "training" / "bbox_detection"),
        name="yolo26n_basketball_player",
        plots=True,
    )
    print(results)
else:
    print("RUN_TRAINING=False；課堂預設使用已訓練 detector 權重進行推論。")


In [ ]:
from src.video_utils import display_video_in_notebook
from src.yolo_utils import write_detection_preview_video

video_candidates = sorted((COURSE_ROOT / "assets" / "raw" / "reference_videos").glob("*.mp4"))
if not video_candidates:
    raise FileNotFoundError("找不到參考影片。請確認 assets/raw/reference_videos/ 內至少有一支 mp4。")

video_path = video_candidates[0]
preview_path = COURSE_ROOT / "assets" / "results" / "d2_03_trained_detector_preview.mp4"
preview_path, records = write_detection_preview_video(
    video_path=video_path,
    model_path=MODEL_PATH,
    output_path=preview_path,
    max_frames=120,
    conf=0.25,
    imgsz=960,
)
preview_json = preview_path.with_suffix(".json")

print("video:", video_path)
print("preview video:", preview_path)
print("preview json:", preview_json)
print("records:", len(records))
display_video_in_notebook(preview_path, loop=True)


## 本單元產出檔案

- `assets/results/d2_03_trained_detector_preview.mp4`：已訓練 detector 的 preview 影片。
- `assets/results/d2_03_trained_detector_preview.json`：preview 逐 frame detection 摘要。
- 若 `RUN_TRAINING=True`，會在 `assets/results/training/bbox_detection/` 看到訓練輸出。
